# 01 — Data Pipeline
**Goal**: Load earnings call transcripts → build event metadata → download market data
→ compute Abnormal Returns, CAR, and ΔVol.

This notebook calls functions from `src/nasdaq_nlp/` and shows the outputs at each step.

In [3]:
import os
from pathlib import Path

# Move to project root so relative paths work (notebooks/ is one level down)
os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()))


## Step 1: Load Transcripts

We have 188 earnings call transcripts across 10 NASDAQ firms (2016–2020).
Each file is a Thomson Reuters StreetEvents (newspaper agency) document in plain text format.

In [4]:
from nasdaq_nlp.data.loader import scan_transcripts, count_by_ticker

records = scan_transcripts()  # returns list[TranscriptRecord]
print(f"Total transcripts: {len(records)}")
print("By ticker:", count_by_ticker(records))

Total transcripts: 188
By ticker: {'AAPL': 19, 'AMD': 19, 'AMZN': 19, 'ASML': 19, 'CSCO': 19, 'GOOGL': 19, 'INTC': 19, 'MSFT': 19, 'MU': 17, 'NVDA': 19}


## Step 2: Event Metadata

For the event study to work, we need to know exactly which **trading day**
the market could react to each call. We call this the event (i.e., earnings call) metadata

**The rule** (US markets):
- If the call is before 4 PM Eastern Time → same-day event
- If the call is at/after 4 PM ET (after market close) → next business day

We parse the timestamp from each transcript header and apply this rule.

In [6]:
from nasdaq_nlp.data.metadata import build_event_metadata
import pandas as pd

meta = build_event_metadata()
print(f"Events: {len(meta)}")
print()
# Show time-of-day distribution
print("Calls after market close:", meta['after_market_close'].sum(), "/", len(meta))
print()
meta[['ticker','year','quarter','call_time_et','after_market_close','event_trading_day']].head(10)

Found 188 transcripts across 10 tickers
Saved event metadata → /Users/javierdominguezsegura/Academics/College/Courses/NLP/NASDAQ-NLP/outputs/processed/event_metadata.csv  (188 rows)
Events: 188

Calls after market close: 169 / 188



,ticker,year,quarter,call_time_et,after_market_close,event_trading_day
0,AAPL,2016,Q1,17:00,True,2016-01-27
1,AAPL,2016,Q2,17:00,True,2016-04-27
2,AAPL,2016,Q3,17:00,True,2016-07-27
3,AAPL,2016,Q4,17:00,True,2016-10-26
4,AAPL,2017,Q1,17:00,True,2017-02-01
5,AAPL,2017,Q2,17:00,True,2017-05-03
6,AAPL,2017,Q3,17:00,True,2017-08-02
7,AAPL,2017,Q4,17:00,True,2017-11-03
8,AAPL,2018,Q1,17:00,True,2018-02-02
9,AAPL,2018,Q2,17:00,True,2018-05-02


## Step 3: Market Data and Returns

We download daily Adjusted Close prices from Yahoo Finance (yfinance) for all 10 tickers
and the NASDAQ Composite index (^IXIC).

**Math — Simple daily return:**

$$R_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$

where $P_t$ is the Adjusted Close price on day $t$.
We use *Adjusted* Close (accounts for splits and dividends) so corporate actions
don't create fake return spikes.

In [ ]:
from nasdaq_nlp.data.market import build_market_returns

# This downloads data if not already cached
stocks, index = build_market_returns()
print(f"Stock return rows: {len(stocks)} | Index return rows: {len(index)}")
print(f"Date range: {stocks['date'].min().date()} → {stocks['date'].max().date()}")
print()
stocks.groupby('ticker')['return'].describe().round(4)

## Step 4: Market Model (OLS), Abnormal Returns, CAR, and ΔVol

**The Market Model (OLS):**

For each earnings call event $i$, we fit a linear regression in the
estimation window (trading days −120 to −20 relative to the call):

$$R_{i,t} = \alpha_i + \beta_i R_{m,t} + \varepsilon_{i,t}$$

- $R_{i,t}$ = stock return on trading day $t$
- $R_{m,t}$ = NASDAQ index return (market return)
- $\alpha_i$ = stock's idiosyncratic excess return
- $\beta_i$ = market sensitivity (e.g. $\beta > 1$ → amplifies market moves)
- $\varepsilon_{i,t}$ = unexplained residual

**Abnormal Return:**

$$AR_{i,t} = R_{i,t} - (\hat{\alpha}_i + \hat{\beta}_i R_{m,t})$$

The *abnormal return* is what the stock returned *beyond* what the market
model predicted — i.e. the earnings-call "surprise."

**Cumulative Abnormal Return (CAR):**

$$\text{CAR}_i[0,3] = AR_{i,0} + AR_{i,1} + AR_{i,2} + AR_{i,3}$$

We sum ARs over a multi-day window to capture the full market reaction
(markets can react over multiple sessions).

**Volatility Change:**

$$\Delta\text{Vol}_i = \sigma(R_{i,+1..+10}) - \sigma(R_{i,-10..-1})$$

Captures whether the call *increased* or *decreased* return variability.

In [ ]:
from nasdaq_nlp.models.market_model import build_event_study

event_study = build_event_study()
print(f"Events with complete data: {event_study['car_03'].notna().sum()}")
print()
print("Summary of CAR[0,3] and ΔVol:")
event_study[['car_01','car_03','ar_0','delta_vol']].describe().round(4)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# CAR[0,3] distribution
axes[0].hist(event_study['car_03'].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', alpha=0.7, label='Zero')
axes[0].set_title('Distribution of CAR[0,3]')
axes[0].set_xlabel('Cumulative Abnormal Return (0→3 days)')
axes[0].set_ylabel('Count')
axes[0].legend()

# ΔVol distribution
axes[1].hist(event_study['delta_vol'].dropna(), bins=30, color='coral', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='steelblue', linestyle='--', alpha=0.7, label='Zero')
axes[1].set_title('Distribution of ΔVolatility')
axes[1].set_xlabel('Post-vol − Pre-vol (std of returns)')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/results/distributions.png', bbox_inches='tight')
plt.show()
print("Saved → outputs/results/distributions.png")

## Verification
Check that all 188 events have complete data (no NaN CAR values).

In [ ]:
nan_car = event_study['car_03'].isna().sum()
nan_vol = event_study['delta_vol'].isna().sum()
assert nan_car == 0, f"FAIL: {nan_car} events have NaN CAR[0,3]"
assert len(event_study) == 188, f"FAIL: expected 188 events, got {len(event_study)}"
print(f"✓ All {len(event_study)} events have complete CAR and ΔVol data")
print(f"✓ {nan_vol} events with NaN ΔVol (expected 0)")